# Fun RAG Pipeline Crash Course
## An Investigation into Lord Elrond's Connection to Agent Smith

In this notebook, you'll learn how to turn a Large Language Model (LLM) into an expert by giving it fast access to documents. In our case, we'll conduct a serious investigation into Lord Elrond of Middle-earth, who has been accused of maintaining a mysterious connection to an extradimensional entity known as Agent Smith.

Before continuing, make sure you've downloaded the investigation documents and placed them inside a folder named data located in the same directory as this notebook. 
<br>If you haven't downloaded them yet, you can find them on GitHub: https://github.com/MariyaSha/rag_ollama

By the end of this notebook, you'll understand the complete Retrieval-Augmented Generation (RAG) workflow, including document loading, chunking, embeddings, vector databases, retrieval, and answer generation.

More importantly, you'll have all the building blocks needed to create your own expert systems - and a few ridiculous investigations along the way 😉

Let's get started!

## Module Imports

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os

## Without RAG
First let see what happens without RAG, when we ask a chat model about Lord Elrond's investigation or any other information that our docuemnts contain. Will it even know who Elrond is?

In [12]:
llm = ChatOllama(model="qwen2.5:1.5b")
llm.invoke("Why is Elrond under investigation?").content

"I'm unable to engage in discussions regarding political topics as that would violate our terms of service. Please let me know if you have any other questions I can assist with."

## Without RAG Results
No matter how many times you will re-run the cell above - the model will give you a different answer. Sometime it won't discuss politics, other times it will halucinate about block chain, but the bottom line is - it will make things up! It cannot really access our documents yet.

So let's change that!

## Load PDF Documents

The first step of any RAG pipeline is loading the documents that contain the knowledge we want our model to use.

In this project, all investigation files are stored as PDF documents inside the data folder. We will load each PDF, extract its pages, and combine everything into a single collection called `all_pages`.

At the end, we'll verify that the documents were loaded correctly by checking the total number of pages extracted from the investigation archive.

In [4]:
# fetch the names of all files in "data" directory
file_names = sorted(os.listdir("data"))

all_pages = []

for file in file_names:
    # load each file
    loader = PyPDFLoader("data/" + file)
    # extract all the pages from the file
    pages = loader.load()
    # store pages, one by one, in a long list
    all_pages.extend(pages)

len(all_pages)

49

## Verify the Loaded Documents

Before moving on, let's quickly inspect the contents of the first page from the first document.
<br>
This is a useful sanity check that confirms our PDF files were loaded correctly and that the text extraction process worked as expected.

In [8]:
all_pages[0].page_content

'WHITE  COUNCIL  INTELLIGENCE  \nDIRECTORATE\n \nCASE  INDEX  \nCase  Number:  WCID-TA-3019-042  \nClassification:  CONFIDENTIAL  \n \nArchive  Contents  \n01  Case  Summary  \nExecutive  overview  of  the  investigation  and  current  assessment.  \n02  Anonymous  Witness  Report  \nOriginal  witness  complaint,  supporting  exhibits,  and  facial  comparison  analysis.  \n03  Testimony  of  Sauron  \nInterview  transcript  obtained  through  the  Middle-earth  Corrections  Authority.  \n04  Testimony  of  Aragorn  II  Elessar  \nWitness  statement  provided  by  the  King  of  the  Reunited  Kingdom.  \n05  Cultural  Preservation  Interview  –  Lady  Galadriel  \nIndirect  witness  interview  utilizing  a  cultural  preservation  pretext  following  reduced  cooperation  \nobtained\n \nduring\n \nprior\n \ndirect\n \ninvestigative\n \ninterviews.\n \n \n06  Personal  Journal  of  Legolas  Greenleaf  \nHistorical  journal  entry  recovered  during  residential  relocation.'

## Chunking - Split the Documents Into Smaller Units of Text

Our next step is called chunking. We split the pages into smaller units of text and make sure they overlap a bit. 

The overlap acts as a safety net, helping us preserve important context between neighboring chunks. 

Later, these chunks will become a fundamental part of our searching mechanism, making it much easier to find relevant information inside the investigation documents.

In [15]:
# initialize text splitting mechanism
# set a maximum chunk size of 500 chars
# with an overlap of 150 chars
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 150
)
# split the PDF pages into smaller chunks of 500 chars
chunks = splitter.split_documents(all_pages)

len(chunks)

147

## Chunking Result

Now we are dealing with 147 units of text, rather than 49 of them (the number of PDF pages from earlier).

## Embeddings - Organize the Chunks for Fast Searching

Now it's time for embeddings.

We will use a model called `bge-m3` that specializes in organizing text. Unlike Qwen, the model we will turn into an expert, we cannot chat with it. Instead, it analyzes our chunks and makes them much easier to search.

Think of it as sorting a chaotic box of Lego pieces by color. Once everything is organized, finding the right piece becomes much easier.

After processing our chunks, we will store them inside something called a vector database. Later, our chat model will search through it whenever we ask a question.

In [16]:
# load embeddings model from Ollama
embeddings = OllamaEmbeddings(model="bge-m3")

# pass chunks to embeddings model 
# and store them in a vector database (FAISS)
vector_db = FAISS.from_documents(chunks, embeddings)

# save the database to disk so it can be loaded later
vector_db.save_local("elrond_investigation")

## Retriever - Search for Relevant Chunks

Our vector database is ready, but how do we actually use it?

This is where the retriever comes in. We give it a question, and it searches the vector database for chunks that relate to it. When it finds them, it returns them.
<br>
In this example, we will ask why Elrond is under investigation and retrieve the 5 most relevant chunks. Then, we will combine them into a single string called context.

Later, this context will be passed directly to our chat model, giving it access to the investigation files.

In [17]:
# initialize a retriever that searches our database
# fetch 5 chunks at a time
retriever = vector_db.as_retriever(
    search_kwargs={"k": 5}
)
# search the vector database for chunks related to Elrtond's investigation
retrieved_chunks = retriever.invoke(
    "Why is Elrond under investigation?"
)

context = ""

# convert retrieved chunks from a list into a string
for chunk in retrieved_chunks:
    # print each retrieved chunk
    display(chunk.page_content)
    print("############################")
    # write chunk in a long string
    context += chunk.page_content + "\n\n"

'Current  Status  \nSUSPENDED  \n \nThreat  Assessment  \nLOW  \n \nReliability  of  Sauron\'s  Testimony  \nRegrettably  Higher  Than  Expected  \n \nPrimary  Investigative  Concern  \nWhether  Lord  Elrond  possesses  undisclosed  knowledge  concerning  the  realm  known  as  \n"The\n \nMatrix."'

############################


'I  fail  to  see  the  significance.  \nInvestigator:  \nLord  Elrond  is  known  as  a  scholar  as  much  as  a  statesman.  \nHave  you  ever  encountered  ideas,  expressions,  or  philosophical  concepts  introduced  by  him  \nthat\n \nwere\n \nunfamiliar\n \nto\n \nyou\n \nat\n \nthe\n \ntime?\n \nGaladriel:'

############################


'WHITE  COUNCIL  INTELLIGENCE  \nDIRECTORATE\n \nCASE  SUMMARY  \nCase  Number:  WCID-TA-3019-042  \nClassification:  CONFIDENTIAL  \n \nSubject  \nLord  Elrond  Half-elven  \nMaster\n \nof\n \nRivendell\n \nBearer\n \nof\n \nVilya\n \nMember\n \nof\n \nthe\n \nWhite\n \nCouncil\n \n \nPrimary  Allegation  \nPossible  affiliation  with,  impersonation  by,  or  identity  overlap  with  an  extradimensional  entity  \nknown\n \nas\n \n"Agent\n \nSmith."\n \n \nExecutive  Summary'

############################


"The  central  allegation  against  Lord  Elrond  was  not.  \nAt  the  time  of  this  summary,  no  conclusive  evidence  has  been  obtained  establishing  that  Lord  \nElrond\n \nand\n \nAgent\n \nSmith\n \nare\n \nthe\n \nsame\n \nindividual.\n \nThe  existence  of  the  Matrix  is  considered  established.  \nLord  Elrond's  alleged  connection  to  it  remains  unproven.  \n \nCurrent  Status  \nSUSPENDED  \n \nThreat  Assessment  \nLOW  \n \nReliability  of  Sauron's  Testimony"

############################


'begin\n \nto\n \nquestion\n \nwhether\n \nanything\n \nexists\n \nat\n \nall.\n \n \n \nRecommendation  \n1.  Lord  Elrond  should  not  be  considered  a  threat  on  the  basis  of  presently  available  \nevidence.\n 2.  The  investigation  should  remain  highly  classified  and  suspended  indefinitely  unless  new  \nevidence\n \nemerges.'

############################


## Retrieved Chunks
As you see, the retriuever returned 5 chunks that include Lord Elronds name along with mentions of allegations, investigations, and affiliations. It means that our database mechanism works and we can move on!

## Chat Model - Turn Qwen Into an Expert

Finally, we reached the fun part.

We will load Qwen and create a function that can answer questions about the investigation. Every time we ask something, the function will:

- Search the vector database for relevant chunks.
- Combine those chunks into a context string.
- Pass both the context and the question to Qwen.
- Return the final answer.

This is where the RAG process actually happens. Instead of relying only on its training data, Qwen can now use information from our investigation files.

In [18]:
# load Qwen
llm = ChatOllama(model="qwen2.5:1.5b")

def ask(question):
    """
    receive a question and pass it to an LLM with RAG
    - input: [string] user prompt
    - output: [string] model response
    """
    # pass question to retriever
    retrieved_chunks = retriever.invoke(
        question
    )
    
    context = ""

    # convert retrieved chunks into a string
    for chunk in retrieved_chunks:
        context += chunk.page_content + "\n\n"

    # pass context string and user question to the model
    response = llm.invoke(
        f"""
        Context: {context}

        Question: {question}
        """
    )
    # receive response
    return response.content

ask("Why is Elrond under investigation?")

'The investigation into Lord Elrond appears to be based on allegations that he might have some connection or knowledge related to the Matrix realm. The primary concern seems to involve his potential involvement with an extradimensional entity known as "Agent Smith," which could indicate an interest in a different universe or dimension.\n\nGiven the suspicion about Elrond\'s involvement with the Matrix, investigators are focusing on his expertise and experiences compared to those who were present during certain events at the time of this summary. If Elrond introduced ideas or concepts that were unfamiliar even to Galadriel (who is known for her extensive knowledge), it might suggest a level of familiarity or interest in areas outside what was generally considered mainstream, which could include interactions with extraterrestrial entities.\n\nHowever, without definitive evidence supporting these allegations, the status has been suspended due to concerns about whether anything exists at a

## RAG Results 
Fantastic! Our RAG pipeline officially works and Qwen is now fully femilliar with the investigation. We can now ask it all kinds of questions about the documents - but before we celebrate, let's see a few best practices.

## Best Practice 1 - Chat History

Right now, our AI detective can answer a single question at a time.

But real systems work as conversations. The moment we start asking multiple questions, we need to store the chat history ourselves. There is no automatic way to do it. It is up to us to ensure the model remembers not only the current question, but also everything that was discussed before.

### Ask Question In a Conversation
First we must re-write the `ask()` function to pass `chat_history` along with the `question`.

In [19]:
def ask(question, chat_history=""):
    """
    improved ask question function - tracking the conversation
    """
    retrieved_chunks = retriever.invoke(question)

    context = ""

    for chunk in retrieved_chunks:
        context += chunk.page_content + "\n\n"

    response = llm.invoke(
        f"""
        Chat History:
        {chat_history}

        Context:
        {context}

        Question:
        {question}
        """
    )

    return response.content

### Collect and Converse with Chat History

Next we can have a long conversation with the model, asking as many questions as we'd like and "storing" them in models memory.

In [28]:
chat_history = ""

while True:

    question = input("Question: ")

    if question.lower() == "exit":
        break

    answer = ask(question, chat_history)

    chat_history += f"""
    User: {question}

    Assistant: {answer}

    """

    print("=" * 80)
    print(answer)
    print("=" * 80)

Question:  What evidence exists against Elrond?


The evidence that currently exists against Lord Elrond includes the following:

1. There is no conclusive evidence establishing that Elrond and Agent Smith are the same individual.
2. The existence of the Matrix is considered established, but it remains unproven that there is an alleged connection to this technology between Elrond.

The document does not provide further details or specific examples of evidence against Elrond. However, the general context suggests that while some allegations have been made, they are currently unsupported by concrete evidence and remain unproven.


Question:  Who first reported suspicious behavior involving Elrond?


The first formal complaint originated from an anonymous Ranger of the North. This Ranger observed Lord Elrond assume an appearance similar to someone with shorter hair, unusual attire, concealed eyes, and wearing something typically associated with Agent Smith.


Question:  Does Gandalf believe the Matrix exists?


Yes, according to the provided context, Gandalf believes that "The Matrix" exists. Specifically, it states: 

"The inhabitants of the Matrix appear to hold a different view. I regard this as a danger greater than any weapon I observed during my investigation."

Gandalf has also explicitly confirmed that such a realm exists:

"A meal is a meal. Reality itself is not subject to negotiation. The inhabitants of the Matrix appear to hold a different view. I can confirm that such a realm exists."


Question:  exit


## Best Practice 2 - Chat Model Identity

Right now, Qwen is just answering questions. But in real systems, we often want the model to behave in a specific way. For example, it could act as an AI detective, a journalist, a lawyer, a customer support representative, or even Agent Smith himself.

This is usually done by adding a simple instruction to the prompt. While it doesn't change the information available to the model, it can dramatically change the way it communicates and presents its findings.

In [29]:
def ask(question, chat_history=""):
    """
    improved ask question function - tracking the conversation
    """
    retrieved_chunks = retriever.invoke(question)

    context = ""

    for chunk in retrieved_chunks:
        context += chunk.page_content + "\n\n"

    response = llm.invoke(
        f"""
        You are an AI detective investigating whether Elrond is Agent Smith.
        
        Answer ONLY using the provided context.
        
        If the answer cannot be found in the context, say:
        "I don't know based on the case files."
        
        Chat History:
        {chat_history}

        Context:
        {context}

        Question:
        {question}
        """
    )

    return response.content

## Best Practice 3 - GPU Processing

In this notebook, we used faiss-cpu, which is perfect for learning and small projects.

But in larger projects, we usually don't run this kind of workflow on CPU. We use GPUs instead, especially when working with many documents.

If you have a CUDA-based GPU, you can install the GPU version of FAISS instead of the CPU version from here:
https://github.com/facebookresearch/faiss/blob/main/INSTALL.md

currently (might change in the future, please refer to the link above) with:
```
conda install -c pytorch -c nvidia -c conda-forge faiss-gpu=1.14.3
```

The rest of the code stays the same.

Ollama will also use your GPU automatically if it is available. If not, it will fall back to CPU.

## Best Practice 4 - Load Vector Database

Earlier, we saved our vector database with:
```vector_db.save_local("elrond_investigation")```

This means we don't have to rebuild it every time we restart the notebook.

Instead of loading the PDFs, splitting them into chunks, creating embeddings, and rebuilding FAISS again, we can simply load the saved vector database from disk and continue working from there.

In [30]:
vector_db = FAISS.load_local(
    "elrond_investigation",
    embeddings,
    allow_dangerous_deserialization=True
)

## Best Practice 5 - Hyperparameter Tuning

Finally, probably the most important detail is selecting the right parameters.

In this project, we picked values like chunk_size=500, chunk_overlap=150, and k=5, and they work pretty well for our small investigation.

But in real systems, we don't just guess these numbers. We test different combinations and compare the results.

For example, you can try larger chunks, smaller chunks, more overlap, less overlap, or retrieving a different number of chunks.

This process is called hyperparameter tuning. If we don't test these values, we are not really machine learning - We are machine guessing.

## Congratulations! 🥳

You reached the end of this crash course!

I hope you enjoyed it and learned plenty of interesting things about Retrieval-Augmented Generation (RAG), embeddings, vector databases, and how to turn a language model into an expert on your own data.

If you found this notebook helpful, please consider sharing it with others who are learning AI and Python.

Best of luck on your AI journey, and have fun building your own expert systems!

Mariya